##  Import Libraries & Load the data

In [1]:
!pip install openai tqdm

In [2]:
import pandas as pd
import openai
import json
from tqdm import tqdm
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


### Data Cleaning

In [3]:
import pandas as pd
df_check = pd.read_excel('all_models_zeroshot.xlsx')
print(df_check.columns.tolist())
print(len(df_check))
df_check.head()

['question', 'GPT-4o Output', 'Allam Output', 'LLaMA Output']
499


,question,GPT-4o Output,Allam Output,LLaMA Output
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","**الإجابة**:\n\n[{""Drug"": ""..."", ""Risk"": ""حاول...","[{""Symptom"": ""أذي نفسي، أفكار بإنهاء حياتي، مح..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ...","```json\n[{""Symptom"": ""تكرار كلمة معينة لمدة ط...","[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب وفوبيا من المجتمع وانعزال...","**الإجابة**:\n\n[{""Symptom"": ""اكتئاب\n"", ""Diag...","[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال...","الإجابة:\n[{""Symptom"": ""قلق شديد مزمن وخوف كبي...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير، وسواس ق..."


In [4]:
all_models = pd.read_excel('all_models_zeroshot.xlsx')

all_models.rename(columns={"LLaMA Output":"LLaMA", "Allam Output": "ALLaM", "GPT-4o Output":"GPT-4o"}, inplace=True)
all_models.columns

Index(['question', 'GPT-4o', 'ALLaM', 'LLaMA'], dtype='object')

In [5]:
import re
from ast import literal_eval


expected_keys = ["Symptom", "Diagnosis", "Treatment", "Drug", "Dosage", "Duration", "Strength", "Unit", "Risk"]

def clean_model_output(text):
    try:
        if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)
        text = text.strip()
        text = re.sub(r'^.*?\[', '[', text, flags=re.DOTALL)
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if not match:
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)
        json_str = match.group(0)
        try:
            parsed = literal_eval(json_str)
        except Exception:
            parsed = json.loads(json_str)
        if isinstance(parsed, list):
            cleaned = []
            for entry in parsed:
                if isinstance(entry, dict):
                    cleaned_dict = {k: str(entry.get(k, "")).strip() for k in expected_keys}
                    cleaned.append(cleaned_dict)
            return json.dumps(cleaned, ensure_ascii=False)
        else:
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)
    except Exception:
        return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

In [6]:
df = all_models.copy()

In [7]:
import re

def clean_allam_output(text: str) -> str:
    s = str(text).strip()
    s = re.sub(r"^(?:الإخراج|Output)\s*:\s*", "", s, flags=re.IGNORECASE)
    if not s:
        return "no answer"
    m = re.search(r"\[\s*\{.*?\}\s*\]", s, flags=re.DOTALL)
    if m:
        return m.group(0)
    start_idx = s.rfind("[{")
    if start_idx != -1:
        candidate = s[start_idx:].strip()
        close_match = re.search(r"\}\s*\]", candidate, flags=re.DOTALL)
        if close_match:
            end = close_match.end()
            return candidate[:end]
        return candidate + "}]"
    return "no answer"

In [10]:
df.columns.tolist()

['question',
 'GPT-4o',
 'ALLaM',
 'LLaMA',
 'ALLaM Cleaned',
 'LLaMA Cleaned',
 'GPT-4o Cleaned']

In [8]:
df["ALLaM Cleaned"] = df["ALLaM"].apply(clean_allam_output)
df["LLaMA Cleaned"] = df["LLaMA"].apply(clean_model_output)
df["GPT-4o Cleaned"] = df["GPT-4o"].apply(clean_model_output)

In [9]:
df.to_excel("zero_all_cleaned.xlsx", index=False, engine='openpyxl')
from google.colab import files
files.download("zero_all_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
zero_all = pd.read_excel('zero_all_cleaned.xlsx')

In [12]:
len(zero_all)

499

In [13]:
zero_all.tail()

,question,GPT-4o,ALLaM,LLaMA,ALLaM Cleaned,LLaMA Cleaned,GPT-4o Cleaned
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","**الإجابة**:\n\n[{""Symptom"": ""قلق, توتر, وسواس...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق, توتر, وسواس قهري, تأتأة في ...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الخوف"", ""Diagnosis"": ""اضطراب الق...","[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل""، ""D...","[{""Symptom"": ""الخوف"", ""Diagnosis"": ""اضطراب الق...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","**الإجابة**:\n[{""Symptom"": "" الذهانية"", ""Diagn...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": "" الذهانية"", ""Diagnosis"": "" خمس س...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",...","الإجابة:\n\n[{""Symptom"": ""فقدان التركيز, النوم...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","[{""Symptom"": ""فقدان التركيز, النوم وقت العمل, ...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك...","```json\n[{""Symptom"": ""الاكتئاب الجزئي، عسر ال...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، فقد...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، عدم...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك..."


In [14]:
zero_all.columns

Index(['question', 'GPT-4o', 'ALLaM', 'LLaMA', 'ALLaM Cleaned',
       'LLaMA Cleaned', 'GPT-4o Cleaned'],
      dtype='object')

In [15]:
df = zero_all[['question', 'LLaMA Cleaned', 'ALLaM Cleaned', 'GPT-4o Cleaned']]

In [16]:
df.head()

,question,LLaMA Cleaned,ALLaM Cleaned,GPT-4o Cleaned
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي، أفكار بإنهاء حياتي، مح...","[{""Drug"": ""..."", ""Risk"": ""حاولت انتحر باكثر من...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،...","[{""Symptom"": ""اكتئاب\n"", ""Diagnosis"": ""فوبيا م...","[{""Symptom"": ""اكتئاب وفوبيا من المجتمع وانعزال..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير، وسواس ق...","[{""Symptom"": ""قلق شديد مزمن وخوف كبير، الوسواس...","[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال..."


## Define the Judge Prompt

### Latest Prompt

In [17]:
def build_prompt(question_text, ner_output):
    return f"""
    You are an expert medical evaluator. Your task is to act as an impartial judge and evaluate the quality of named entity extraction performed by a single AI model. We will provide you with a medical question,
    a model-generated list of extracted entities, and a set of evaluation criteria. You should first read the question carefully, then evaluate the extracted entities based on the following two criteria: Correctness and Completeness.

    For each criterion:
    - Use only the question text as the source of truth.
    - Refer strictly to the entity definitions provided.
    - Do not infer or hallucinate missing information.
    - Output your final verdict using one of the valid options for each criterion (see below).
    - Provide a short explanation for your decision.

    ## Question:
    {question_text}

    ## Model Output (NER Prediction):
    {ner_output}

    ## Entity Schema:
    Evaluate only the following entity types and definitions:
    - Symptom: Observable psychological or behavioral symptoms (e.g., anxiety, insomnia).
    - Diagnosis: Mentioned or diagnosed mental health disorders (e.g., generalized anxiety disorder).
    - Treatment: Non-pharmacological therapy methods (e.g., cognitive behavioral therapy).
    - Drug: Names of medications or pharmaceutical substances (e.g., Xanax, Sertraline).
    - Dosage: Dosage instructions (e.g., twice a day, 50 mg).
    - Duration: Duration of medication use or symptom persistence (e.g., for one week, for 3 months).
    - Strength: Concentration of the medication (e.g., 500, 1.0).
    - Unit: Unit of measurement for strength (e.g., mg, ml, g).
    - Risk: Indicators of psychological or physical risk (e.g., suicidal thoughts, self-harm).

    ## Evaluation Criteria

    [The Start of Correctness Criterion]
    Does the model correctly extract the entity spans and assign the correct entity labels **based only on the question text**?

    - accurate: Most or all entities are correct, with minor or no issues.
    - mixed: Some correct extractions alongside boundary or label errors.
    - poor: Many incorrect, misaligned, mislabeled, or spurious entities.
    [The End of Correctness Criterion]

    [The Start of Completeness Criterion]
    How well did the model cover all important entities mentioned **in the question**?

    - full: All relevant entities present in the question are extracted.
    - partial: Some entities are extracted (about half or more).
    - low: Less than half of the important entities are captured.
    [The End of Completeness Criterion]

    ## Output Format
    Please return your evaluation in the following exact format as a markdown-style JSON block:

    {{
      "correctness": {{
        "verdict": "accurate/mixed/inaccurate",
        "reason": "Your detailed reasoning here..."
      }},
      "completeness": {{
        "verdict": "full/partial/missing",
        "reason": "Your detailed reasoning here..."
      }},
      "notes": "Optional comments if needed"
    }}

    """


## DeepSeek

In [18]:
os.environ["DEEPSEEK_API_KEY"] = "sk-771606b2d6ad468d8ee3f15fe3f165e9"

In [19]:
import re
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

def score_with_deepseek(prompt):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0
        )
        content = response.choices[0].message.content.strip()

        # Try to extract JSON block
        match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", content, re.DOTALL)
        if not match:
            match = re.search(r"(\{.*\})", content, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        else:
            raise ValueError("No JSON block found")

    except Exception as e:
        return {
            "correctness": {"verdict": "error", "reason": str(e)},
            "completeness": {"verdict": "error", "reason": str(e)},
            "notes": ""
        }

In [21]:
# To test connection
test_prompt = build_prompt(df.iloc[0]['question'], df.iloc[0]['LLaMA Cleaned'])
print(score_with_deepseek(test_prompt))

{'correctness': {'verdict': 'accurate', 'reason': "The model correctly identified 'الخوف من عدم الإنجاب' (fear of not having children) as a Symptom, which aligns with the definition of observable psychological symptoms. The span is accurate and the label is appropriate based on the question text."}, 'completeness': {'verdict': 'partial', 'reason': "The model extracted the main symptom but missed other relevant entities such as the intensity of the fear ('خايفة جداً' - very afraid) which could be considered part of the symptom description, and the context of being 'على وجه جواز' (newly married) which is not an entity per schema. However, no other entity types (Diagnosis, Treatment, Drug, etc.) are explicitly mentioned in the question, so the only missing element is the severity qualifier which is not a separate entity type. Thus, completeness is partial as the core symptom is captured but not all nuances."}, 'notes': 'The question is in Arabic and the model output is in Arabic. The extr

In [22]:
# Iterate and collect results
model_cols = ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]

In [23]:
from collections import Counter

# Reset before each new run
summary = {col: {"correctness": Counter(), "completeness": Counter()} for col in model_cols}
results = {col: [] for col in model_cols}

In [24]:
from tqdm import tqdm

for col in model_cols:
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Scoring {col}"):
        prompt = build_prompt(row['question'], row[col])
        score = score_with_deepseek(prompt)

        results[col].append(score)

        # ✅ Update verdict counts
        if isinstance(score, dict):
            c = score.get("correctness", {}).get("verdict", "error")
            cm = score.get("completeness", {}).get("verdict", "error")
            summary[col]["correctness"][c] += 1
            summary[col]["completeness"][cm] += 1

Scoring GPT-4o Cleaned: 100%|██████████| 499/499 [16:59<00:00,  2.04s/it]


In [25]:
results_dict = {}

## LLaMA

In [26]:
from tqdm import tqdm
import json

model_col = "LLaMA Cleaned"
judgment_field = "LLaMA Judgment"


for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {
            model_col: row[model_col]
        }

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save after full column is done
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)


Evaluating LLaMA Cleaned: 100%|██████████| 499/499 [17:29<00:00,  2.10s/it]


In [27]:
llama = pd.DataFrame(results_dict.values())

In [28]:
llama.tail()

,LLaMA Cleaned,LLaMA Judgment
494,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ..."
495,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ..."
496,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso..."
497,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ..."


In [29]:
llama.insert(0, 'Question', df['question'])

In [30]:
llama.head()

,Question,LLaMA Cleaned,LLaMA Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي، أفكار بإنهاء حياتي، مح...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ...","{'correctness': {'verdict': 'accurate', 'reaso..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،...","{'correctness': {'verdict': 'accurate', 'reaso..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير، وسواس ق...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [31]:
llama.to_excel("zero_llama_judgments.xlsx", index=False)

from google.colab import files
files.download("zero_llama_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Allam

In [32]:
import json
from tqdm import tqdm

model_col = "ALLaM Cleaned"
judgment_field = "ALLaM Judgment"

# Load existing results
with open("ner_judgment_results.json", "r", encoding="utf-8") as f:
    results_dict = json.load(f)

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Re-evaluating {model_col}"):
    key = row["question"]

    # Initialize model output in results if missing
    if key not in results_dict:
        results_dict[key] = {}
    results_dict[key][model_col] = row[model_col]

    # Always re-run ALLaM's judgment
    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save updated JSON
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Re-evaluating ALLaM Cleaned: 100%|██████████| 499/499 [17:51<00:00,  2.15s/it]


In [33]:
from google.colab import files
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
allam = pd.DataFrame(results_dict.values())

In [35]:
allam.insert(0, 'Question', df['question'])

In [36]:
allam.tail()

,Question,LLaMA Cleaned,LLaMA Judgment,ALLaM Cleaned,ALLaM Judgment
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""قلق, توتر, وسواس قهري, تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""الخوف"", ""Diagnosis"": ""اضطراب الق...","{'correctness': {'verdict': 'poor', 'reason': ..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": "" الذهانية"", ""Diagnosis"": "" خمس س...","{'correctness': {'verdict': 'poor', 'reason': ..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""فقدان التركيز, النوم وقت العمل, ...","{'correctness': {'verdict': 'poor', 'reason': ..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، عدم...","{'correctness': {'verdict': 'poor', 'reason': ..."


In [ ]:
# allam.to_excel("final_judgments.xlsx", index=False)

# from google.colab import files
# files.download("final_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
allam.to_excel("zero_allam_judgments.xlsx", index=False)
files.download("zero_allam_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# import pandas as pd


# # Loop through each question entry in results_dict
# for question, data in results_dict.items():
#     for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
#         # Find the corresponding judgment field for that model
#         judgment_field = model_col.replace("Cleaned", "Judgment").strip()

#         if judgment_field in data:
#             score = data[judgment_field]
#             if isinstance(score, dict):
#                 flattened.append({
#                     "Question": question,
#                     "Model": model_col,
#                     "Correctness": score["correctness"]["verdict"],
#                     "Correctness Reason": score["correctness"]["reason"],
#                     "Completeness": score["completeness"]["verdict"],
#                     "Completeness Reason": score["completeness"]["reason"],
#                     "Notes": score.get("notes", "")
#                 })

# # Create a DataFrame and export
# df_results = pd.DataFrame(flattened)
# df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
# print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

Exported 1497 rows to NER_Evaluation_by_DeepSeek.xlsx


In [ ]:
# df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The question only mentions a psychological sym...,"The model performed well in this case, as the ..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured some key entities like 'Sym...,The model's performance is mixed due to some r...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,accurate,The model correctly identifies and labels the ...,full,The model captures all relevant entities menti...,The model performs well in both correctness an...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified and labeled all...,full,The model extracted all the important entities...,The model performed well in both correctness a...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,accurate,The model correctly extracted and labeled the ...,full,The model captured all relevant entities menti...,The model performed well in both correctness a...


In [ ]:
# df_results['Model'].unique()

array(['LLaMA Cleaned', 'ALLaM Cleaned'], dtype=object)

In [ ]:
# files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## GPT-4o

In [39]:
model_col = "GPT-4o Cleaned"
judgment_field = "GPT-4o Judgment"

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {model_col: row[model_col]}
    else:
        results_dict[key][model_col] = row[model_col]

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Evaluating GPT-4o Cleaned: 100%|██████████| 499/499 [15:29<00:00,  1.86s/it]


In [40]:
gpt = pd.DataFrame(results_dict.values())

In [41]:
gpt.insert(0, 'Question', df['question'])

In [42]:
gpt.head()

,Question,LLaMA Cleaned,LLaMA Judgment,ALLaM Cleaned,ALLaM Judgment,GPT-4o Cleaned,GPT-4o Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'mixed', 'reason':..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي، أفكار بإنهاء حياتي، مح...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Drug"": ""..."", ""Risk"": ""حاولت انتحر باكثر من...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ...","{'correctness': {'verdict': 'mixed', 'reason':..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""اكتئاب\n"", ""Diagnosis"": ""فوبيا م...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""اكتئاب وفوبيا من المجتمع وانعزال...","{'correctness': {'verdict': 'mixed', 'reason':..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير، وسواس ق...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق شديد مزمن وخوف كبير، الوسواس...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [43]:
gpt.tail()

,Question,LLaMA Cleaned,LLaMA Judgment,ALLaM Cleaned,ALLaM Judgment,GPT-4o Cleaned,GPT-4o Judgment
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""قلق, توتر, وسواس قهري, تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""الخوف"", ""Diagnosis"": ""اضطراب الق...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","{'correctness': {'verdict': 'mixed', 'reason':..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": "" الذهانية"", ""Diagnosis"": "" خمس س...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""فقدان التركيز, النوم وقت العمل, ...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، عدم...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [44]:
gpt.to_excel("zero_gpt_judgments.xlsx", index=False)

from google.colab import files
files.download("zero_gpt_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
import json
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)
from google.colab import files
files.download("ner_judgment_results.json")

import pandas as pd
from collections import Counter

correctness_scale = {"accurate": 2, "mixed": 1, "poor": 0, "inaccurate": 0}
completeness_scale = {"full": 2, "partial": 1, "low": 0, "missing": 0}
model_fields = {"GPT-4o": "GPT-4o Judgment", "LLaMA": "LLaMA Judgment", "ALLaM": "ALLaM Judgment"}

rows = []
for model_name, field in model_fields.items():
    c_counts = Counter(); p_counts = Counter(); c_scores, p_scores = [], []
    for key, data in results_dict.items():
        judgment = data.get(field)
        if not isinstance(judgment, dict):
            continue
        c_verdict = judgment.get("correctness", {}).get("verdict", "error")
        p_verdict = judgment.get("completeness", {}).get("verdict", "error")
        c_counts[c_verdict] += 1; p_counts[p_verdict] += 1
        if c_verdict in correctness_scale: c_scores.append(correctness_scale[c_verdict])
        if p_verdict in completeness_scale: p_scores.append(completeness_scale[p_verdict])
    rows.append({
        "Model": model_name,
        "Correctness Avg": round(sum(c_scores)/len(c_scores), 3) if c_scores else None,
        "Completeness Avg": round(sum(p_scores)/len(p_scores), 3) if p_scores else None,
        "Correctness Counts": dict(c_counts), "Completeness Counts": dict(p_counts),
        "N errors": c_counts.get("error", 0),
    })
summary_df = pd.DataFrame(rows)
summary_df

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Model,Correctness Avg,Completeness Avg,Correctness Counts,Completeness Counts,N errors
0,GPT-4o,1.264,1.024,"{'mixed': 358, 'accurate': 135, 'poor': 3, 'er...","{'partial': 261, 'full': 124, 'low': 108, 'mis...",2
1,LLaMA,1.127,1.248,"{'accurate': 115, 'mixed': 329, 'poor': 37, 'e...","{'full': 191, 'partial': 237, 'missing': 28, '...",3
2,ALLaM,0.738,0.812,"{'mixed': 316, 'inaccurate': 17, 'accurate': 2...","{'partial': 287, 'low': 121, 'missing': 30, 'f...",3


In [46]:
import pandas as pd


# Loop through each question entry in results_dict
for question, data in results_dict.items():
    for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
        # Find the corresponding judgment field for that model
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()

        if judgment_field in data:
            score = data[judgment_field]
            if isinstance(score, dict):
                flattened.append({
                    "Question": question,
                    "Model": model_col,
                    "Correctness": score["correctness"]["verdict"],
                    "Correctness Reason": score["correctness"]["reason"],
                    "Completeness": score["completeness"]["verdict"],
                    "Completeness Reason": score["completeness"]["reason"],
                    "Notes": score.get("notes", "")
                })

# Create a DataFrame and export
df_results = pd.DataFrame(flattened)
df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

NameError: name 'flattened' is not defined

In [47]:
df_results.head()

NameError: name 'df_results' is not defined

In [ ]:
df_results['Model'].unique()

array(['LLaMA Cleaned', 'ALLaM Cleaned', 'GPT-4o Cleaned'], dtype=object)

In [ ]:
files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>